# Reflection profile from Julia time-domain data

This notebook reads the CSV profile selected by `DT` and produces a reflection-vs-photon-frequency figure.

In [41]:
from pathlib import Path
import os


def find_project_dir():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        direct = candidate
        nested = candidate / "timedomain_cross_validation"

        for path in (direct, nested):
            if (path / "jl_src" / "reflection.jl").exists():
                return path

    raise FileNotFoundError("Could not find timedomain_cross_validation from the current working directory.")


PROJECT_DIR = find_project_dir()
MPLCONFIGDIR = PROJECT_DIR / ".mplconfig"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "lines.linewidth": 1.0,
    "lines.markersize": 4,
})

pi = np.pi
RESOLUTION = 300


In [55]:
CSV_DIR = PROJECT_DIR / "results" / "csv"
FIG_DIR = PROJECT_DIR / "results" / "fig"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Choose the time step whose CSV/figure pair you want to use.
DT = 2.8

# The Julia model is written in a rotating frame: only Delta = omega_A - omega_p is simulated.
# These values define the absolute frequency axis used for comparison with the Python simulations.
param_cavity = {"omega_A": 10*pi, "gamma_A": pi, "L": 100}

PROJECT_DIR

PosixPath('/Users/romainpiron/Documents/NII_local/waveguide-qed-simulator/timedomain_cross_validation')

In [56]:
def dt_label(dt):
    return str(dt).replace("/", "_over_").replace(" ", "")


def candidate_dt_labels(dt):
    labels = [dt_label(dt)]
    if isinstance(dt, (int, np.integer)):
        labels.append(f"{dt}.0")
    if isinstance(dt, (float, np.floating)) and float(dt).is_integer():
        labels.append(f"{float(dt):.1f}")
    return list(dict.fromkeys(labels))


def reflection_csv_for_dt(dt):
    for label in candidate_dt_labels(dt):
        csv_path = CSV_DIR / f"reflection_profile_jl_dt_{label}.csv"
        if csv_path.exists():
            return label, csv_path

    available = sorted(path.name for path in CSV_DIR.glob("reflection_profile_jl_dt_*.csv"))
    raise FileNotFoundError(
        f"No CSV found for DT={dt!r} in {CSV_DIR}. Available files: {available}"
    )


DT_LABEL, CSV_PATH = reflection_csv_for_dt(DT)
PDF_PATH = FIG_DIR / f"reflection_profile_jl_dt_{DT_LABEL}_result.pdf"

df = pd.read_csv(CSV_PATH).sort_values("Delta_over_gamma")
DT_FROM_CSV = df["dt"].iloc[0] if "dt" in df.columns else DT

photon_frequency_tab = param_cavity["omega_A"] - df["Delta_over_gamma"].to_numpy()*param_cavity["gamma_A"]
final_reflection_tab = df["reflection"].to_numpy()

sort_index = np.argsort(photon_frequency_tab)
photon_frequency_tab = photon_frequency_tab[sort_index]
final_reflection_tab = final_reflection_tab[sort_index]

CSV_PATH


PosixPath('/Users/romainpiron/Documents/NII_local/waveguide-qed-simulator/timedomain_cross_validation/results/csv/reflection_profile_jl_dt_2.8.csv')

In [57]:
fig, ax = plt.subplots(figsize=(3, 3), dpi=RESOLUTION)

color_bare = "#3E4A89"
color_data = "#a82402"

frequency_tab_theory = np.linspace(
    param_cavity["omega_A"] - param_cavity["gamma_A"]/2,
    param_cavity["omega_A"] + param_cavity["gamma_A"]/2,
    100,
)
R_theory = 1 / (1 + ((frequency_tab_theory - param_cavity["omega_A"]) / (param_cavity["gamma_A"] / 2))**2)

ax.scatter(
    photon_frequency_tab,
    final_reflection_tab,
    label=r"$\mathcal{R}$ (WaveguideQED.jl)",
    color=color_data,
    s=20,
)

ax.plot(
    frequency_tab_theory,
    R_theory,
    color=color_bare,
    alpha=0.8,
    zorder=-1,
    label=r"$\mathcal{R}^{(\rm th)}$",
)

ax.set_xlabel(r"\textbf{Photon frequency} $\omega_p$", fontsize=10)
ax.set_xticks([
    param_cavity["omega_A"] - param_cavity["gamma_A"]/2,
    param_cavity["omega_A"] - param_cavity["gamma_A"]/4,
    param_cavity["omega_A"],
    param_cavity["omega_A"] + param_cavity["gamma_A"]/4,
    param_cavity["omega_A"] + param_cavity["gamma_A"]/2,
])
ax.set_xticklabels([r"$9.5\pi$", r"$9.75\pi$", r"$10\pi$", r"$10.25\pi$", r"$10.5\pi$"])

ax.set_ylabel(r"\textbf{Probability}", fontsize=10)
ax.set_ylim(-0.05, 1.05)
ax.grid(color="0.9", linestyle="-", linewidth=0.4)
ax.legend(prop={"size": 10}, loc="lower center", frameon=False)

for item in [ax.xaxis.label, ax.yaxis.label]:
    item.set_fontsize(10)

for item in (ax.get_xticklabels() + ax.get_yticklabels()):
    item.set_fontsize(10)

fig.tight_layout()

fig.savefig(PDF_PATH, bbox_inches="tight")
PDF_PATH


/var/folders/n2/ld5ngfbx4891xn96x63rbjlm0000gn/T/ipykernel_11797/3267882011.py:1: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(3, 3), dpi=RESOLUTION)


PosixPath('/Users/romainpiron/Documents/NII_local/waveguide-qed-simulator/timedomain_cross_validation/results/fig/reflection_profile_jl_dt_2.8_result.pdf')